In [1]:
from finvizfinance.earnings import Earnings
import pandas as pd
from datetime import datetime, timedelta
import re


In [10]:
# Load the full month of earnings once
e = Earnings()
e._set_period('This Month')
df = e.df.copy()

/Users/devna/personalProjects/financeApp/venv/lib/python3.13/site-packages/finvizfinance/screener/base.py:134: FutureWarning: The behavior of DataFrame concatenation with empty or all-NA entries is deprecated. In a future version, this will no longer exclude empty or all-NA columns when determining the result dtypes. To retain the old behavior, exclude the relevant entries before the concat operation.
  return pd.concat([df, pd.DataFrame(frame)], ignore_index=True)
/Users/devna/personalProjects/financeApp/venv/lib/python3.13/site-packages/finvizfinance/screener/base.py:134: FutureWarning: The behavior of DataFrame concatenation with empty or all-NA entries is deprecated. In a future version, this will no longer exclude empty or all-NA columns when determining the result dtypes. To retain the old behavior, exclude the relevant entries before the concat operation.
  return pd.concat([df, pd.DataFrame(frame)], ignore_index=True)


In [17]:
# Ensure required columns exist
required_cols = ['Ticker', 'Market Cap', 'Earnings']
for col in required_cols:
    if col not in df.columns:
        raise RuntimeError(f"Expected column '{col}' not found. Columns: {df.columns}")

# Convert Market Cap to numeric for sorting
def parse_market_cap(mc):
    if pd.isna(mc):
        return 0
    mc = str(mc).upper().strip()
    if mc.endswith('T'):
        return float(mc[:-1]) * 1_000_000_000_000
    elif mc.endswith('B'):
        return float(mc[:-1]) * 1_000_000_000
    elif mc.endswith('M'):
        return float(mc[:-1]) * 1_000_000
    elif mc.endswith('K'):
        return float(mc[:-1]) * 1_000
    else:
        try:
            return float(mc)
        except:
            return 0

df['Market Cap Numeric'] = df['Market Cap'].apply(parse_market_cap)


def parse_earnings_date(s):
    if pd.isna(s):
        return None
    try:
        return datetime.strptime(f"{s} {datetime.today().year}", "%b %d %Y")
    except:
        return None
    
df['Earnings Clean'] = df['Earnings'].str.replace(r'/[ab]$', '', regex=True)

df['Earnings Date'] = df['Earnings Clean'].apply(parse_earnings_date)


def format_market_cap(n):
    if n >= 1_000_000_000_000:
        return f"{n/1_000_000_000_000:.2f}T"
    elif n >= 1_000_000_000:
        return f"{n/1_000_000_000:.2f}B"
    elif n >= 1_000_000:
        return f"{n/1_000_000:.2f}M"
    elif n >= 1_000:
        return f"{n/1_000:.2f}K"
    else:
        return str(n)

df['Market Cap Formatted'] = df['Market Cap Numeric'].apply(format_market_cap)


In [45]:
today = datetime.today()

current_week_start = today - timedelta(days=today.weekday())
current_week_end = current_week_start + timedelta(days=6)

def filter_current_week(df):
    filtered = df[(df['Earnings Date'] >= current_week_start) & (df['Earnings Date'] <= current_week_end)]
    # Include the parsed Earnings Date for later use
    return filtered.sort_values('Market Cap Numeric', ascending=False).head(10)[
        ['Ticker', 'Market Cap Formatted', 'Earnings', 'Earnings Date']
    ]


def filter_this_week(df):
    start = today
    end = today + timedelta(days=7)
    filtered = df[(df['Earnings Date'] >= start) & (df['Earnings Date'] <= end)]
    return filtered.sort_values('Market Cap Numeric', ascending=False).head(10)[['Ticker', 'Market Cap Formatted', 'Earnings']]

def filter_next_week(df):
    start = today + timedelta(days=7)
    end = today + timedelta(days=14)
    filtered = df[(df['Earnings Date'] >= start) & (df['Earnings Date'] <= end)]
    return filtered.sort_values('Market Cap Numeric', ascending=False).head(10)[['Ticker', 'Market Cap Formatted', 'Earnings']]

def filter_rest_of_month(df):
    start = today
    end = df['Earnings Date'].max()
    filtered = df[(df['Earnings Date'] >= start) & (df['Earnings Date'] <= end)]
    return filtered.sort_values('Market Cap Numeric', ascending=False)[['Ticker', 'Market Cap Formatted', 'Earnings']]


In [46]:
filter_current_week(df).head(10)

,Ticker,Market Cap Formatted,Earnings,Earnings Date
1952,CSCO,308.35B,Nov 12/a,2025-11-12
2082,DIS,190.22B,Nov 13/b,2025-11-13
2345,MUFG,180.32B,Nov 14/a,2025-11-14
2230,AMAT,180.05B,Nov 13/a,2025-11-13
1757,SONY,180.03B,Nov 11/b,2025-11-11
1763,SONY,180.03B,Nov 11/b,2025-11-11
2344,SMFG,111.56B,Nov 14/a,2025-11-14
2340,MFG,87.26B,Nov 14/a,2025-11-14
2339,MFG,87.26B,Nov 14/a,2025-11-14
2164,NU,76.23B,Nov 13/a,2025-11-13


In [30]:
# Top companies this week
filter_this_week(df).head(10)

,Ticker,Market Cap Formatted,Earnings
2443,NVDA,4.62T,Nov 19/a
2470,WMT,817.06B,Nov 20/b
2390,HD,360.69B,Nov 18/b
2491,INTU,184.68B,Nov 20/a
2402,PDD,181.86B,Nov 18/b
2423,TJX,162.58B,Nov 19/b
2444,PANW,140.39B,Nov 19/a
2421,LOW,127.78B,Nov 19/b
2405,MDT,122.97B,Nov 18/b
2397,MDT,122.97B,Nov 18/b


In [78]:
# Top companies next week
filter_next_week(df).head(10)

KeyError: 'Market Cap Numeric'

In [32]:
# Top companies rest of the month
filter_rest_of_month(df).head(10)

,Ticker,Market Cap Formatted,Earnings
2443,NVDA,4.62T,Nov 19/a
2470,WMT,817.06B,Nov 20/b
2390,HD,360.69B,Nov 18/b
2527,BABA,345.04B,Nov 25/b
2491,INTU,184.68B,Nov 20/a
2402,PDD,181.86B,Nov 18/b
2423,TJX,162.58B,Nov 19/b
2444,PANW,140.39B,Nov 19/a
2555,DE,128.74B,Nov 26/b
2421,LOW,127.78B,Nov 19/b


In [37]:
tickers = filter_current_week(df)['Ticker'].unique().tolist()
tickers

['CSCO', 'DIS', 'MUFG', 'AMAT', 'SONY', 'SMFG', 'MFG', 'NU']

In [39]:
import yfinance as yf
from datetime import timedelta

# Example: last 5 trading days before and 1 day after earnings
def get_price_history(ticker, earnings_date, days_before=5, days_after=1):
    start = earnings_date - timedelta(days=days_before)
    end = earnings_date + timedelta(days=days_after)
    
    data = yf.download(ticker, start=start, end=end)
    return data


In [56]:
price_history = {}

current_week_df = filter_current_week(df)

for idx, row in current_week_df.iterrows():
    ticker = row['Ticker']
    earnings_date = row['Earnings Date']  # parsed datetime

    start = earnings_date - timedelta(days=5)
    end = earnings_date + timedelta(days=1)

    try:
        data = yf.download(ticker, start=start, end=end)
        if not data.empty:
            price_history[ticker] = data
        else:
            print(f"No data returned for {ticker} around {earnings_date.date()}")
    except Exception as e:
        print(f"Error downloading {ticker}: {e}")

# Check which tickers worked
price_history.keys()


/var/folders/tm/6q9t2fd55k7776dc86cxzb5c0000gn/T/ipykernel_16050/3351313149.py:13: FutureWarning: YF.download() has changed argument auto_adjust default to True
  data = yf.download(ticker, start=start, end=end)
[*********************100%***********************]  1 of 1 completed
/var/folders/tm/6q9t2fd55k7776dc86cxzb5c0000gn/T/ipykernel_16050/3351313149.py:13: FutureWarning: YF.download() has changed argument auto_adjust default to True
  data = yf.download(ticker, start=start, end=end)
[*********************100%***********************]  1 of 1 completed
/var/folders/tm/6q9t2fd55k7776dc86cxzb5c0000gn/T/ipykernel_16050/3351313149.py:13: FutureWarning: YF.download() has changed argument auto_adjust default to True
  data = yf.download(ticker, start=start, end=end)
[*********************100%***********************]  1 of 1 completed
/var/folders/tm/6q9t2fd55k7776dc86cxzb5c0000gn/T/ipykernel_16050/3351313149.py:13: FutureWarning: YF.download() has changed argument auto_adjust default to T

dict_keys(['CSCO', 'DIS', 'MUFG', 'AMAT', 'SONY', 'SMFG', 'MFG', 'NU'])

In [57]:
price_history['CSCO']
csco = price_history['CSCO']
csco['Pct Change'] = csco['Close'].pct_change() * 100
csco


Price,Close,High,Low,Open,Volume,Pct Change
Ticker,CSCO,CSCO,CSCO,CSCO,CSCO,
Date,,,,,,
2025-11-07,71.070000,71.589996,70.540001,71.389999,16913700,NaN
2025-11-10,72.089996,72.500000,71.150002,71.760002,22201200,1.435200
2025-11-11,71.709999,72.250000,71.099998,71.739998,21412900,-0.527115
2025-11-12,73.959999,74.209999,71.720001,71.910004,57591600,3.137638


In [63]:
import pandas as pd
import numpy as np

# Dictionary to hold percent changes
pct_changes_dict = {}
dates_list = []

# First, collect all dates from all tickers
for ticker, hist in price_history.items():
    if 'Pct Change' not in hist.columns:
        hist['Pct Change'] = hist['Close'].pct_change() * 100
    
    pct_changes = hist['Pct Change'].round(2).tolist()
    pct_changes_dict[ticker] = pct_changes

    # Store dates
    dates_list.append(hist.index.strftime('%Y-%m-%d').tolist())

# Find the maximum length
max_len = max(len(d) for d in dates_list)

# Pad shorter tickers with NaN so all lists have the same length
for ticker in pct_changes_dict:
    current_len = len(pct_changes_dict[ticker])
    if current_len < max_len:
        pct_changes_dict[ticker] += [np.nan] * (max_len - current_len)

# Use the first ticker’s dates padded to max_len as column names
all_dates = dates_list[0] + [f'ExtraDay{i+1}' for i in range(max_len - len(dates_list[0]))]

# Build DataFrame
pct_changes_df = pd.DataFrame.from_dict(pct_changes_dict, orient='index')
pct_changes_df.columns = all_dates

pct_changes_df


,2025-11-07,2025-11-10,2025-11-11,2025-11-12,ExtraDay1
CSCO,NaN,1.44,-0.53,3.14,NaN
DIS,NaN,2.33,1.57,-7.75,NaN
MUFG,NaN,-0.13,2.30,-0.64,2.27
AMAT,NaN,-2.73,0.90,-3.25,NaN
SONY,NaN,-2.05,0.94,4.25,NaN
SMFG,NaN,0.24,1.09,-0.72,5.49
MFG,NaN,0.30,1.04,-0.15,3.83
NU,NaN,2.31,-1.28,-3.65,NaN


In [71]:
import yfinance as yf
import pandas as pd

ticker = "CSCO"
csco = yf.Ticker(ticker)

# Get earnings dates (historical + upcoming)
earnings = csco.earnings_dates

# Filter for 2023 and later
earnings = earnings[earnings.index.year >= 2023]

# Optionally only take last 10 earnings
earnings = earnings.tail(10)

# Function to calculate price reaction
def price_reaction(ticker, date):
    date = pd.to_datetime(date)
    # Grab the day of and previous day prices
    hist = yf.download(ticker, start=date - pd.Timedelta(days=1),
                       end=date + pd.Timedelta(days=2), progress=False)
    if hist.shape[0] >= 2:
        # Reaction from close before earnings to close after earnings
        before_close = hist['Close'].iloc[0]
        after_close = hist['Close'].iloc[-1]
        return round((after_close - before_close) / before_close * 100, 2)
    return None

# Add price reaction column
earnings['Price Reaction (%)'] = [price_reaction(ticker, d) for d in earnings.index]

# Reset index for nicer display
earnings = earnings.reset_index()
earnings = earnings.rename(columns={'index': 'Release Date'})

print(earnings)


              Earnings Date  EPS Estimate  Reported EPS  Surprise(%)  \
0 2025-05-14 16:00:00-04:00          0.92          0.96         4.64   
1 2025-02-12 16:00:00-05:00          0.91          0.94         3.41   
2 2024-11-13 16:00:00-05:00          0.87          0.91         4.47   
3 2024-08-14 16:00:00-04:00          0.85          0.87         2.49   
4 2024-05-15 16:00:00-04:00          0.82          0.88         7.71   
5 2024-02-14 16:00:00-05:00          0.84          0.87         3.96   
6 2023-11-15 16:00:00-05:00          1.03          1.11         7.71   
7 2023-08-16 16:00:00-04:00          1.06          1.14         7.68   
8 2023-05-17 16:00:00-04:00          0.97          1.00         3.25   
9 2023-02-15 16:00:00-05:00          0.86          0.88         2.84   

                    Price Reaction (%)  
0   Ticker
CSCO    2.98
dtype: float64  
1   Ticker
CSCO    3.91
dtype: float64  
2   Ticker
CSCO   -2.13
dtype: float64  
3   Ticker
CSCO    9.01
dtype: float64  
4 

/var/folders/tm/6q9t2fd55k7776dc86cxzb5c0000gn/T/ipykernel_16050/3005097654.py:20: FutureWarning: YF.download() has changed argument auto_adjust default to True
  hist = yf.download(ticker, start=date - pd.Timedelta(days=1),
/var/folders/tm/6q9t2fd55k7776dc86cxzb5c0000gn/T/ipykernel_16050/3005097654.py:20: FutureWarning: YF.download() has changed argument auto_adjust default to True
  hist = yf.download(ticker, start=date - pd.Timedelta(days=1),
/var/folders/tm/6q9t2fd55k7776dc86cxzb5c0000gn/T/ipykernel_16050/3005097654.py:20: FutureWarning: YF.download() has changed argument auto_adjust default to True
  hist = yf.download(ticker, start=date - pd.Timedelta(days=1),
/var/folders/tm/6q9t2fd55k7776dc86cxzb5c0000gn/T/ipykernel_16050/3005097654.py:20: FutureWarning: YF.download() has changed argument auto_adjust default to True
  hist = yf.download(ticker, start=date - pd.Timedelta(days=1),
/var/folders/tm/6q9t2fd55k7776dc86cxzb5c0000gn/T/ipykernel_16050/3005097654.py:20: FutureWarning: YF

In [75]:
# Make historical index tz-naive
hist.index = hist.index.tz_localize(None)
reactions = []
for date in eps_df["Earnings Date"]:
    next_day = pd.to_datetime(date) + pd.Timedelta(days=1)
    pos = hist.index.searchsorted(next_day)
    if pos < len(hist):
        trading_day = hist.index[pos]
        reactions.append(hist.loc[trading_day, 'Pct Change'])
    else:
        reactions.append(None)

eps_df["Price Reaction (%)"] = reactions
eps_df["Price Reaction (%)"] = eps_df["Price Reaction (%)"].round(2)
print(eps_df)



  Earnings Date  EPS Estimate  Reported EPS  Price Reaction (%)
0    2025-05-14          0.92          0.96                4.85
1    2025-02-12          0.91          0.94                2.09
2    2024-11-13          0.87          0.91               -2.13
3    2024-08-14          0.85          0.87                6.80
4    2024-05-15          0.82          0.88               -2.68
5    2024-02-14          0.84          0.87               -2.43
6    2023-11-15          1.03          1.11               -9.83
7    2023-08-16          1.06          1.14                3.34
8    2023-05-17          0.97          1.00                1.20
9    2023-02-15          0.86          0.88                5.24
